# Lab 3 — Inteligencia Artificial Explicable (xAI) en monitoreo estructural

Entrenamos un clasificador **XGBoost** multiclass sobre datos de sensores de monitoreo de salud estructural (SHM) y aplicamos un **kit completo de técnicas xAI** sobre ese mismo modelo: importancia del booster, permutation importance, **SHAP** (global y local), **LIME** y **PDP**. El objetivo no es solo predecir el estado de una estructura — es poder **explicar y auditar** esa predicción antes de confiar en ella para una alerta de daño.

El notebook recorre: carga y limpieza del dataset → balance de clases → entrenamiento del XGBoost → métricas de clasificación → explicación global (importancia, permutation) → explicación local de un caso concreto con SHAP y LIME (comparadas lado a lado) → efecto marginal de una feature con PDP y SHAP dependence.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (labs/.venv).')

## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración en tres ejes |
| Strain | με | Deformación (extensómetro) |
| Temp | °C | Temperatura del sensor |
| **Condition Label** | **0 / 1 / 2** | **Normal / daño menor / severo** |

Mismo CSV que **Lab 1**. Entrenamos **XGBoost** y aplicamos un **kit xAI**: importancia, permutation, **SHAP**, **LIME** y **PDP**.

Detalle: [`data/DATOS.md`](data/DATOS.md).

## 1. Panorama xAI en monitoreo estructural

El objetivo central del lab **no es solo entrenar** el clasificador: es **probar varias técnicas de explicación** sobre el mismo modelo. xAI muestra **qué sensores** empujaron una predicción, mientras que accuracy solo dice si el modelo acertó — y en obra eso no basta: antes de activar una alerta de daño, el ingeniero necesita saber *por qué* el modelo la disparó.

### Caja de herramientas xAI (este lab)

| Técnica | Alcance | Sección | Idea clave |
|---------|---------|---------|------------|
| **Importancia del booster** | Global | 7 | Qué sensores usa el árbol en promedio |
| **Permutation importance** | Global | 7 (pre) | Cuánto cae el rendimiento si barajas una feature |
| **SHAP** (`TreeExplainer`) | Global + local | 8–9 | Contribución marginal por sensor (eficiente en árboles) |
| **LIME** | Local | 10 | Aproximación interpretable de **un caso** (comparar con SHAP) |
| **PDP + SHAP dependence** | Global marginal | 11 | Efecto promedio de un sensor vs interacciones |

SHAP es estable y rápido en árboles; LIME es flexible y útil para comparar un caso concreto con lenguaje simple — en la sección 10 las ponemos una junto a la otra sobre el mismo caso. Global = comportamiento promedio del modelo; local = una lectura de sensor en un instante concreto.

In [ ]:
TECNICAS_XAI = ["importancia", "permutation", "shap", "lime", "pdp"]
print("Técnicas xAI del lab:")
for t in TECNICAS_XAI:
    print(f"  · {t}")

## 2. Carga del dataset de sensores

El CSV trae 5 features de sensor (aceleración en 3 ejes, deformación, temperatura), un `Timestamp` y la etiqueta `Condition Label`. `Timestamp` ordena las lecturas en el tiempo pero no entra al modelo — no aporta información sobre el estado estructural en sí.

In [ ]:
# --- PRE-ESCRITO: carga ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")

In [ ]:
FEATURES = [
    "Accel_X (m/s^2)", "Accel_Y (m/s^2)", "Accel_Z (m/s^2)",
    "Strain (με)", "Temp (°C)",
]
N_FILAS_HEAD = 5
print(f"Features: {FEATURES}")
display(df.head(N_FILAS_HEAD))

## 3. Calidad de datos y limpieza

Eliminamos filas con valores nulos en cualquiera de las 5 features de sensor: 96 lecturas se pierden (1000 → 904). Antes de limpiar, vale la pena revisar el sensor más relevante para daño estructural — `Strain (με)` — en sus datos crudos, para tener una referencia de sus estadísticas originales.

In [ ]:
# --- PRE-ESCRITO: limpieza ---
n_antes = len(df)
n_nulos_por_col = df[FEATURES].isna().sum()
print("Nulos por sensor (crudo):")
display(n_nulos_por_col)
df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
print(f"Tras dropna: {n_antes} → {n_despues} lecturas")

In [ ]:
COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas «{COLUMNA_REVISAR}» (crudo):")
display(stats_col)

## 4. Balance de clases (Condition Label)

La clase 0 (normal) es mayoritaria, lo cual es típico en monitoreo estructural — la mayor parte del tiempo la estructura está sana. Por eso usamos `stratify` al dividir train/test: mantiene las mismas proporciones de clase en ambos conjuntos, evitando que el test quede sin ejemplos suficientes de daño severo.

In [ ]:
# --- PRE-ESCRITO: distribución de etiquetas ---
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(conteo).plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_xlabel('Condition Label')
ax.set_ylabel('Conteo')
ax.set_title('Distribución de estados estructurales')
plt.tight_layout()
plt.show()
print("Conteo:", conteo)

In [ ]:
N_CLASES_MOSTRAR = 3
serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
display(serie_clases)

## 5. Partición y entrenamiento de XGBoost

Escalamos los sensores antes de entrenar porque sus magnitudes son muy distintas (`Accel_Z` ronda 9.8 m/s² por la gravedad, mientras `Strain` ronda 80 με) — un StandardScaler evita que una feature domine solo por su escala. `max_depth` controla qué tan complejas pueden ser las interacciones que cada árbol captura, y `learning_rate` cuánto corrige cada árbol nuevo al anterior.

In [ ]:
# --- PRE-ESCRITO: escalado y split estratificado ---
X_raw = df_limpio[FEATURES].values
y = df_limpio['Condition Label'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
N_ESTIMATORS = 100
MAX_DEPTH = 6
LEARNING_RATE = 0.1
modelo = XGBClassifier(
    objective='multi:softprob', num_class=3,
    n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE, random_state=RANDOM_STATE, eval_metric='mlogloss',
)
modelo.fit(X_train, y_train)
print("✅ XGBoost entrenado.")

## 6. Métricas de clasificación

La confusión entre clase 1 (daño menor) y clase 2 (daño severo) es habitual en este tipo de problema — son estados intermedios que comparten patrones de sensor. Para un sistema de alertas, el recall de la clase 2 es el número que más importa: un falso negativo ahí significa no detectar daño severo real.

In [ ]:
# --- PRE-ESCRITO: predicciones en test ---
y_pred = modelo.predict(X_test)
acc_test = accuracy_score(y_test, y_pred)
print(f"Accuracy test: {acc_test:.3f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2], yticklabels=[0, 1, 2], ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión')
plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred, digits=3))

## 7. xAI global (1): importancia del booster y permutation importance

Dos formas de medir importancia global que no son lo mismo: la importancia del booster refleja cuánto usa el árbol cada feature internamente (gain/weight), mientras que permutation importance mide cuánto cae el rendimiento real del modelo si barajas esa columna — una mide estructura interna, la otra impacto medido. Si el top sensor coincide con `Strain`, es una buena señal de coherencia física: el extensómetro es, por diseño, el sensor más ligado al daño estructural.

In [ ]:
# --- PRE-ESCRITO: dos técnicas xAI globales ---
imp_series = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
imp_series.plot(kind='barh', ax=ax, color='#8e44ad')
ax.set_xlabel('Importancia (gain/weight)')
ax.set_title('xAI global — Importancia del booster (XGBoost)')
plt.tight_layout()
plt.show()

perm = permutation_importance(modelo, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE)
perm_series = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
perm_series.plot(kind='barh', ax=ax, color='#16a085')
ax.set_xlabel('Caída media en score al permutar')
ax.set_title('xAI global — Permutation importance')
plt.tight_layout()
plt.show()

In [ ]:
TOP3_IMPORTANCIA = imp_series.sort_values(ascending=False).head(3).index.tolist()
print(f"Top-3: {TOP3_IMPORTANCIA}")

## 8. xAI global (2): SHAP summary plot

SHAP reparte, para cada predicción, cuánto empujó cada feature el resultado hacia una clase u otra — el summary plot agrega eso sobre todo el test set para una clase específica. Explicamos la clase 2 (daño severo), que es la que más importa para una alerta: se espera ver `Strain` con un |SHAP| alto en promedio, confirmando lo que ya sugería la importancia del booster.

In [ ]:
# --- PRE-ESCRITO: TreeExplainer ---
def shap_values_clase(shap_values, clase: int):
    if isinstance(shap_values, list):
        return shap_values[clase]
    return shap_values[:, :, clase]

explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test)
if isinstance(shap_values, list):
    n_clases_shap, n_casos_shap = len(shap_values), shap_values[0].shape[0]
else:
    n_casos_shap, _, n_clases_shap = shap_values.shape
print(f"SHAP: {n_clases_shap} clases × {n_casos_shap} casos test")

In [ ]:
CLASE_SHAP = 2
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values_clase(shap_values, CLASE_SHAP), X_test, feature_names=FEATURES, show=False)
plt.title('SHAP summary — daño severo (clase 2)')
plt.tight_layout(); plt.show()

## 9. xAI local con SHAP (waterfall)

La explicación global dice qué sensores importan *en promedio*; para auditar una predicción concreta necesitamos una explicación local. El waterfall de SHAP descompone una predicción individual, mostrando cuánto empujó cada sensor desde el valor base hasta la probabilidad final — exactamente el tipo de evidencia que iría en un informe de inspección.

In [ ]:
# --- PRE-ESCRITO: utilidad para índice en test ---
def etiqueta_en_test(idx: int) -> tuple[int, int]:
    yt = int(y_test[idx])
    yp = int(y_pred[idx])
    return yt, yp

In [ ]:
INDEX_CASO = 0
y_true_caso, y_pred_caso = etiqueta_en_test(INDEX_CASO)
print(f"real={y_true_caso}, predicho={y_pred_caso}")
clase_explicar = CLASE_SHAP
sv = shap_values_clase(shap_values, clase_explicar)[INDEX_CASO]
base = explainer.expected_value
if isinstance(base, (list, np.ndarray)):
    base = base[clase_explicar]
exp = shap.Explanation(values=sv, base_values=base, data=X_test[INDEX_CASO], feature_names=FEATURES)
shap.plots.waterfall(exp, max_display=6, show=False)
plt.tight_layout(); plt.show()

## 10. xAI local con LIME (mismo caso que SHAP)

**LIME** (Local Interpretable Model-agnostic Explanations) aproxima el modelo **solo alrededor de un caso**, a diferencia de SHAP que reparte de forma exacta la salida del modelo. Usamos el mismo `INDEX_CASO` y `CLASE_SHAP` que en la sección 9 para comparar ambas técnicas directamente: normalmente coinciden en qué sensor es más influyente, aunque LIME puede variar un poco entre corridas porque depende de un muestreo local aleatorio alrededor del caso.

In [ ]:
# --- PRE-ESCRITO: explainer LIME tabular ---
explainer_lime = LimeTabularExplainer(
    X_train,
    feature_names=FEATURES,
    class_names=['Normal (0)', 'Daño menor (1)', 'Daño severo (2)'],
    mode='classification',
    random_state=RANDOM_STATE,
)
print("✅ LIME TabularExplainer listo (fondo = train).")

In [ ]:
exp_lime = explainer_lime.explain_instance(
    X_test[INDEX_CASO], modelo.predict_proba, num_features=5, labels=(CLASE_SHAP,),
)
fig = exp_lime.as_pyplot_figure(label=CLASE_SHAP)
plt.title(f'LIME — caso {INDEX_CASO}')
plt.tight_layout(); plt.show()
TOP_LIME_FEATURES = [
    next((f for f in FEATURES if feat.startswith(f)), feat.split('<=')[0].strip())
    for feat, _ in exp_lime.as_list(label=CLASE_SHAP)[:3]
]
print(f"Top-3 LIME: {TOP_LIME_FEATURES}")

## 11. PDP y SHAP dependence (efecto marginal)

Un PDP (partial dependence plot) muestra el efecto marginal *promedio* de una feature sobre la predicción, variándola mientras las demás se mantienen fijas. El dependence plot de SHAP muestra lo mismo pero caso por caso, lo que además revela interacciones (por ejemplo, coloreando por una segunda feature) que el PDP, al promediar, puede ocultar. Aquí miramos `Temp (°C)` — la celda pre-escrita ya muestra el PDP de `Strain` como referencia.

In [ ]:
# --- PRE-ESCRITO: PDP para Strain ---
idx_strain = FEATURES.index('Strain (με)')
fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(
    modelo, X_test, [idx_strain], feature_names=FEATURES, target=2, ax=ax,
)
ax.set_title('PDP — Strain (με)')
plt.tight_layout()
plt.show()

In [ ]:
FEATURE_PDP = "Temp (°C)"
idx_pdp = FEATURES.index(FEATURE_PDP)
fig, ax = plt.subplots(figsize=(6, 4))
PartialDependenceDisplay.from_estimator(modelo, X_test, [idx_pdp], feature_names=FEATURES, target=CLASE_SHAP, ax=ax)
ax.set_title(f'PDP — {FEATURE_PDP}')
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 4))
shap.dependence_plot(idx_pdp, shap_values_clase(shap_values, CLASE_SHAP), X_test, feature_names=FEATURES, show=False)
plt.title(f'SHAP dependence — {FEATURE_PDP}')
plt.tight_layout(); plt.show()

## Reflexión: preguntas que los alumnos necesitarían

- `Strain` domina tanto la importancia del booster como SHAP — ¿en qué caso *desconfiarías* de esa coincidencia (por ejemplo, si `Strain` estuviera fuertemente correlacionado con otro sensor)?
- Si SHAP y LIME dieran resultados claramente distintos para el mismo caso, ¿a cuál le darías más peso en un informe de inspección, y por qué?
- xAI explica el modelo, no la física del daño real — ¿qué evidencia adicional (normativa, inspección visual, histórico) necesitarías antes de actuar sobre una alerta basada solo en SHAP?
- La clase 2 (daño severo) es minoritaria en el dataset. ¿Cómo podría eso sesgar tanto las métricas del modelo como las explicaciones xAI, y qué harías al respecto?
- Si tuvieras que explicarle el waterfall de SHAP a un colega sin experiencia en ML, ¿qué analogía de ingeniería estructural usarías?